<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 3 · Calibration and Fourier Pricing

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/03_dxquant_calibration_and_fourier.ipynb)


## Notebook Goals

This notebook illustrates a simple parametric calibration loop:

- generate a synthetic European call surface from the Heston (1993) model,
- price calls via the Carr–Madan Fourier approach, and
- recover the original parameters by least-squares calibration.

The focus is on the workflow and the interfaces, not on producing market-grade calibration
diagnostics.

## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## Synthetic Target Surface

We define a small strike grid and generate target prices from a known Heston parameter set. The
resulting prices serve as the calibration target.

In [ ]:
spot = 100.0
rate = 0.01
maturity_years = 1.0

strikes = np.linspace(80.0, 120.0, 9)

target_params = np.array([2.0, 0.04, 0.5, -0.4])
kappa_0, theta_0, sigma_0, rho_0 = target_params

target_process = dx.HestonProcess(
    kappa=kappa_0,
    theta=theta_0,
    sigma=sigma_0,
    rho=rho_0,
    drift=rate,
)

pricer = dx.FourierPricer()
discount = float(np.exp(-rate * maturity_years))

phi_0 = target_process.characteristic_function(
    tau=maturity_years,
    spot=spot,
    variance=theta_0,
)

target_prices = np.array(
    [pricer.price_call(phi_0, spot, strike, discount) for strike in strikes],
    dtype=float,
)

target_prices

## Least-Squares Calibration

The calibration objective maps model parameters to a vector of call prices on the strike grid.
Least-squares calibration minimizes the price differences to the target surface.

In [ ]:
def model_surface(params: np.ndarray) -> np.ndarray:
    kappa, theta, sigma, rho = params
    process = dx.HestonProcess(
        kappa=float(kappa),
        theta=float(theta),
        sigma=float(sigma),
        rho=float(rho),
        drift=rate,
    )
    phi = process.characteristic_function(
        tau=maturity_years,
        spot=spot,
        variance=float(theta),
    )
    prices = [pricer.price_call(phi, spot, k, discount) for k in strikes]
    return np.array(prices, dtype=float)

guess = np.array([1.5, 0.03, 0.6, -0.2])
bounds = (
    np.array([1e-4, 1e-4, 1e-4, -0.99]),
    np.array([5.0, 1.0, 2.0, 0.99]),
)

fitted = dx.calibrate_least_squares(
    model=model_surface,
    target=target_prices,
    guess=guess,
    bounds=bounds,
)

fitted

## Parameter Recovery

We compare fitted parameters and fitted prices to the synthetic targets.

In [ ]:
labels = ["kappa", "theta", "sigma", "rho"]
for label, value, target in zip(labels, fitted, target_params, strict=True):
    diff = float(value - target)
    msg = (
        f"{label:>5s}: fitted={value: .6f}  "
        f"target={target: .6f}  diff={diff: .6f}"
    )
    print(msg)

fitted_prices = model_surface(fitted)
max_abs = float(np.max(np.abs(fitted_prices - target_prices)))
print(f"Max abs price error: {max_abs:.6e}")

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">